# Python Package for the Survey of Consumer Finances
#### Author: Payton Crawford
#### Last Updated: November 18th, 2026

## Table of Contents
* [1. Introduction](#1-introduction)
* [2. Installing scfpublic](#2-installing-scfpublic)
* [3. Importing SCF Data](#3-importing-scf-data)
* [4. Standard Error Adjustments](#4-standard-error-adjustments)
* [5. Examples](#5-examples)
    * [5.1 Historical Tables](#51-historical-tables-comparison)
    * [5.2 Standard Error Documentation](#52-standard-error-documentation-comparison)

## 1. Introduction
Traditionally, the Survey of Consumer Finances (SCF) has been provided publicly as SAS, STATA, and CSV files. The purpose of this Python package, `scfpublic`, is to provide a reliable and replicable way to download SCF data (full data, summary extract data, and replicate weight files) and properly correct for multiple imputation and sampling error to regressions and other statistics. Additionally, it encourages the use of Parquets, which can compress datasets down significantly in comparison to unzipped .dta and .cpt files. This open-source data format is supported by many data analysis programs including Pandas in Python, R, and SAS (Stata added support in StataNow with the release of Stata 19.0 while an alternative could be to use the [pystata](https://www.stata.com/features/overview/pystata/) package). This package enables the user to download dta files directly from the [SCF website](https://www.federalreserve.gov/econres/scfindex.htm) with only year parameters required, format replicate weights, and correct for appropriate error corrections with respect to user-defined functions.


## 2. Installing scfpublic
When using Python, you typically use pip to install packages in your terminal. So, for installing the `scfpublic` package globally, simply run:

In [ ]:
pip install scfpublic

From here, you should be able to call `import scfpublic` in any .py file. For best practices, you can instead run `python -m pip install scfpublic` to ensure the correct version of pip is used for your specific environment.

## 3. Importing SCF Data
Firstly, you will want to import the `scfpublic` package. After this is done, you can simply call the year(s), dataset(s), and variable(s) that you want. For example:

In [1]:
from scfpublic import download_year, download_years, download_main_extr_years

# Providing your selected columns as a string or a list is acceptable (This is an optional parameter, if you do not specify columns, it will download all variables)
columns_str = 'yy1 y1 year wgt income networth edcl x14 x19 x501'
columns_list = ['yy1', 'y1', 'year', 'wgt', 'income', 'networth', 'edcl', 'x14', 'x19', 'x501']

main_df = download_main_extr_years(2025, columns=columns_str, verbose=True)

print(main_df.head())

Reading: p25i6.dta
Loaded: 21,800 rows | 6 columns
Reading: rscfp2025.dta
Loaded: 21,800 rows | 7 columns
Optimizing final merged dataframe...
Initial memory: 0.00 GB
Final memory: 0.00 GB (14.3% reduction)
   x14  x19  x501  y1  yy1  year  edcl         income   networth          wgt
0   39   40     3  11    1  2025     4  317351.351351  2434270.0  1241.701712
1   39   40     3  12    1  2025     4  317351.351351  2208400.0  1247.038677
2   39   40     3  13    1  2025     4  317351.351351  2128300.0  1234.428835
3   39   40     3  14    1  2025     4  317351.351351  1742300.0  1267.530314
4   39   40     3  15    1  2025     4  317351.351351  1517300.0   911.408367


Alternatively, you can call `download_years()` to produce a cleaned merge between years and `download_main_extr_years()` to download a merged main data and bulletin data for the years you specify. WARNING: Downloading many years at once is a memory-intensive process for your RAM, specifically during the merge and concatenation commands. If the program crashes while downloading all dataframes, it might be advisable to download them separately using `download_year()`. Then, you can tamper with your settings to slowly merge the files together or reduce the datasets to only load the variables you need. 

## 4. Standard Error Adjustments
Now that you have your main dataset, any statistical results need to be properly adjusted for imputation and sampling error. For more technical details, see the [Standard Error documentation](https://www.federalreserve.gov/econres/files/Standard_Error_Documentation.pdf). In this case, we can load the replicate weights for 2025 and keep the Pandas dataframes separate for use in `scf_prepare_data()`.

In [2]:
rep_wgts = download_year(2025, file_type='rep_wgts')
print(rep_wgts.head())

   y1         wt1b1         wt1b2         wt1b3         wt1b4         wt1b5  \
0  11           NaN   4867.290318           NaN   5091.516726           NaN   
1  21  88263.690778  57599.143600  66404.751039  80927.190885           NaN   
2  31  22499.693595           NaN  16690.683946           NaN  20894.124225   
3  41    115.282677    191.736539    239.029629    200.790335    173.851763   
4  51  52786.162253  47166.157170  45929.411631           NaN  49149.035665   

          wt1b6         wt1b7         wt1b8         wt1b9  ...  mm8   mm7  \
0   7492.180097   6434.135150   8052.303331   7615.796829  ...    1     1   
1  74389.731565  60641.910077  71618.574828  64017.725876  ...    1     1   
2  13692.351913           NaN  18603.017183  14430.925194  ...    1  <NA>   
3    202.554629    169.225325    201.213212    204.585682  ...    1     3   
4           NaN  49140.908306  50543.341687  44447.045504  ...    1     2   

    mm6   mm5   mm4   mm3   mm2   mm1  yy1  year  
0     2  <N

Now that we have our dataset and our replicate weights, we import the `scfcombo` package and prepare the dataset.

In [3]:
from scfpublic.scfcombo import scfcombo_result, prepare_scf_data, scf_estimate, scf_mean, scf_quantile, scf_regression

scf_data = prepare_scf_data(main_df, rep_wgts)
print(scf_data)

{'implicates': [      x14  x19  x501     y1   yy1  year  edcl        income     networth  \
0      39   40     3     11     1  2025     4  3.173514e+05    2434270.0   
1      69   59     3     21     2  2025     4  6.675676e+04    3005160.0   
2      80    0     3     31     3  2025     4  2.187568e+05   20396060.0   
3      78   72     3     41     4  2025     4  3.376865e+06  107852700.0   
4      42   41     3     51     5  2025     4  1.376216e+05     417500.0   
...   ...  ...   ...    ...   ...   ...   ...           ...          ...   
4355   67   63     3  43751  4375  2025     4  1.704865e+07  274328000.0   
4356   62   50     3  43761  4376  2025     2  6.316216e+05   14988000.0   
4357   28   25     3  43771  4377  2025     2  3.697297e+04      -3320.0   
4358   38    0     3  43781  4378  2025     4  6.162162e+04     343800.0   
4359   62    0     3  43791  4379  2025     4  1.181081e+05     532300.0   

               wgt  
0      1241.701712  
1     13727.793587  
2      3

Now that our data is properly prepared, we can run our summary statistics. For convenience, the wrapper functions `scf_mean`, `scf_quantile`, and `scf_regression` (the default is a weighted least squares model) are provided. First, you will see how the wrapper function works and then an explicit way to use `scf_estimate`. When these functions are run, they return a scfcombo_result data object. This stores the results for later use, and the `display()` command reports back formatted results. If you want to use estimations stored in the scfcombo_result object, you can access its attributes. For example, you can call `result.estimate` to only return the mean below (or an array of coefficients depending on the estimation function).

In [4]:
result = scf_mean(scf_data, variable='income')
result.display()


Mean: income

Estimate                144630.1112
Std. Error                2425.4477
Z-Score                     59.6303
P>|z|                        0.0000

[95% Conf. Interval]      139876.3211       149383.9014

------------------------------------------------------------------------------
Variance Decomposition:
------------------------------------------------------------------------------
Imputation SE                     1308.8487  ( 29.1% of variance)
Sampling SE                       2041.9872  ( 70.9% of variance)
Total SE                          2425.4477

------------------------------------------------------------------------------
Number of observations: 4,360
Number of implicates: 5
Number of replicates: 999



Note that while there are 21,800 observations in the main dataset, there are only 4,360 observations after correctly accounting for imputation and sampling error. Now, an explicit way to use scf_estimate():

In [ ]:
import pandas as pd
import numpy as np

def weighted_mean(data: pd.DataFrame, weights: np.ndarray, variable: str) -> float:
        notmissing = data[variable].notna()
        if not notmissing.any():
            return np.nan
        return np.average(data.loc[notmissing, variable].values, weights=weights[notmissing])

result = scf_estimate(scf_data, weighted_mean, variable='income')
result.statistic_name = 'Mean'
result.variable_name = 'income'

result.display()

As expected, the estimates are exactly the same. If you instead want a dataframe of results for the coefficient(s) of the scf_estimate operation, you can call result.summary() to return a pandas DataFrame with calculated Z-scores, p-values, 95% CIs (to your preference of alpha), and the reported imputation/sampling variances. To export your results, you can save using Pandas' built-in save functions to the data format as your choice, such as `to_parquet`.

## 5. Examples
### 5.1 Historical Tables Comparison
For comparison, you can use the raw extract data to compare to the published Historical Tables. For example, if we replicated Table 5 of the Historical Tables for 2025 inflation-adjusted dollars [here](https://www.federalreserve.gov/econres/files/scf2025_tables_public_real_historical.xlsx), we would download the data between 1989 and 2025. Table 5 estimates the value of financial assets of all families (summed weighted value).

In [ ]:
import scfpublic
import numpy as np
import pandas as pd

vars = (
    "y1 yy1 wgt "
    "fin liq cds nmmf stocks bond retqliq savbnd cashli othma othfin nfin "
    "hfin hliq hcds hnmmf hstocks hbond hretqliq hsavbnd hcashli hothma hothfin hnfin"
)

print(vars)

df = scfpublic.download_years(range(1989, 2026, 3), file_type= 'extr_data', columns=vars)
print(df.head())

y1 yy1 wgt fin liq cds nmmf stocks bond retqliq savbnd cashli othma othfin nfin hfin hliq hcds hnmmf hstocks hbond hretqliq hsavbnd hcashli hothma hothfin hnfin
        bond  cashli     cds       fin  hbond  hcashli  hcds  hfin  hliq  \
312535   0.0   500.0  5000.0  183000.0      0        1     1     1     1   
312536   0.0   500.0  5000.0  183000.0      0        1     1     1     1   
312537   0.0   500.0  5000.0  183000.0      0        1     1     1     1   
312538   0.0   500.0  5000.0  183000.0      0        1     1     1     1   
312539   0.0   500.0  5000.0  183000.0      0        1     1     1     1   

        hnfin  ...  nmmf  othfin  othma  retqliq   savbnd  stocks  \
312535      1  ...   0.0     0.0    0.0  80000.0  25000.0     0.0   
312536      1  ...   0.0     0.0    0.0  80000.0  25000.0     0.0   
312537      1  ...   0.0     0.0    0.0  80000.0  25000.0     0.0   
312538      1  ...   0.0     0.0    0.0  80000.0  25000.0     0.0   
312539      1  ...   0.0     0.0    0

With this data, you can now adjust the variables such that only applicable values will be collected in the sums. For example, if a household does not have financial assets (hfin = 0), then the financial assets variable (fin) is set to missing rather than incorrectly being included as a observation of 0. Then, a weighted sum function and a percentages function are constructed to get the summed percentage for each type of financial asset relative to the total sum.

In [ ]:

vars = "liq cds nmmf stocks bond retqliq savbnd cashli othma fin othfin nfin"
vars_list = vars.split()

# Convert in-applicable values to NaN values
for var in vars_list:
    if f'h{var}' in df:
        df.loc[df[f'h{var}'] == 0, f'{var}'] = np.nan
        df.loc[df[f'{var}'] == 0, f'{var}'] = np.nan
        df.loc[pd.isna(df[f'{var}']), f'h{var}'] = 0

    else:
        print(f'h{var}')

# Define weighted sum calculation
def weighted_sum(df, col, weight_col):
    """Calculate weighted sum for a single column, ignoring NaNs"""
    mask = ~df[col].isna()
    if mask.sum() > 0:
        return (df.loc[mask, col] * df.loc[mask, weight_col]).sum()
    else:
        return np.nan

# Convenience function for dividing weighted percentage by denominator sum
def weighted_percentages(df, numerator_vars, denominator_var, weight_col='wgt'):
    """
    Calculate weighted percentage divided by weighted sum of denominator variable
    Returns array of length = len(numerator_vars)
    """
    # Calculate weighted sum for denominator
    denom_sum = weighted_sum(df, denominator_var, weight_col)
    
    # Calculate weighted percentage for each numerator variable
    percentages = []
    for col in numerator_vars:
        num_sum = weighted_sum(df, col, weight_col)
        percentage = (num_sum / denom_sum) * 100
        percentages.append(percentage)
    
    return np.array(percentages)

Mirroring Table 5, I define row names matching the Table 5 values and append each year's summed values to a Pandas DataFrame.

In [ ]:
vars = "liq cds savbnd bond stocks nmmf retqliq cashli othma othfin fin"
vars_list = vars.split()

percentages_index = [
                    'Transaction accounts',
                    'Certificates of deposit',
                    'Savings bonds',
                    'Bonds',
                    'Stocks',
                    'Pooled investment funds (excluding money market funds)',
                    'Retirement accounts',
                    'Cash value life insurance',
                    'Other managed assets',
                    'Other',
                    'Total',
                    'Financial assets as a share of total assets'
                ]

percentages = pd.DataFrame()
percentages.index = percentages_index

for year in range(1989,2026,3):

    df_cut = df[df['year'] == year]
    # Calculate percentages
    year_df = weighted_percentages(df_cut, vars_list, 'fin', 'wgt')

    fin_percent_assets = weighted_sum(df_cut, 'fin', 'wgt') / (weighted_sum(df_cut, 'fin', 'wgt') + weighted_sum(df_cut, 'nfin', 'wgt'))
    year_df = np.append(year_df, fin_percent_assets * 100)
    percentages[f'{year}'] = year_df

print(percentages.round(1)) 

As you can check by hand, every number matches up exactly. You can replicate every other table of the Historical Tables using this package.

### 5.2 Standard Error Documentation Comparison
To explicitly show the difference in standard errors after accounting for multiple imputation variance and sampling variance, this demo shows how standard errors differ on a base regression. For example, we can take the years 2025 and run a weighted least squares regression on adjusted logged income (log(income + 1)) using homeownership (hhouses) with controls for education (edcl), age (age), and having a mortgage (hmrthel) fixed effects conditional on the household head being aged 25-40.

In [ ]:
from scfpublic import download_year, download_years, download_main_extr_years
from scfpublic.scfcombo import scfcombo_result, prepare_scf_data, scf_regression
import numpy as np
import statsmodels.formula.api as smf

#####
# Run regressions using loaded data
#####
vars = 'yy1 y1 income hhouses hdebt hmrthel edcl age wgt'

df = download_main_extr_years(2025, columns = vars, verbose=True)
rep_wgts = download_years(2025, file_type='rep_wgts')

df['lninc'] = np.log(df['income'] + 1)

cut_df = df[df['age'].between(25,40)]

formula = 'lninc ~ C(edcl) + age + hhouses + hmrthel'

model = smf.wls(formula=formula, weights=cut_df['wgt'], data=cut_df).fit()
print(model.summary2(float_format="%.4f"))

As seen above, this prints out the regression results without accounting for imputation correction or sampling variance correction. Of note is the number of observations between 25 and 40 with all implicates (4,995) and the coefficient (0.1500) and standard errors (0.0461) for hhouses. For this regression, housecl has a z-score of 3.25, which is statistically significant at the 99% confidence level. Now, we run scf_regression, which accounts for both multiple imputation corrections and sampling corrections. Be aware that running scf_regression can take time to run (upwards of 60 seconds).

In [ ]:
# SCF-Combo Regression
scf_data = prepare_scf_data(df, rep_wgts)

result = scf_regression(scf_data, formula = formula, subset=lambda df:  df['age'].between(25, 40))

result.display()

Now, note that housecl is no longer statistically significant with a z-score of 1.55 and a p-value of 0.122. Thus, accounting for imputation and sampling variance can affect the statistical significance of the results. For more information, please view the standard error documentation [here](https://www.federalreserve.gov/econres/files/Standard_Error_Documentation.pdf).